# Data Preprocessing

## Initialization

In [28]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [29]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from copy import deepcopy
from sklearn.preprocessing import OneHotEncoder

import src.utils as utils

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [30]:
from typing import cast

# Train
X_train = cast(pd.DataFrame, utils.deserialize_data("data/interim/X_train.pkl"))
y_train = cast(pd.Series, utils.deserialize_data("data/interim/y_train.pkl"))

# Valid
X_valid = cast(pd.DataFrame, utils.deserialize_data("data/interim/X_valid.pkl"))
y_valid = cast(pd.Series, utils.deserialize_data("data/interim/y_valid.pkl"))

# Test
X_test = cast(pd.DataFrame, utils.deserialize_data("data/interim/X_test.pkl"))
y_test = cast(pd.Series, utils.deserialize_data("data/interim/y_test.pkl"))

## Handling Data Duplication

In [31]:
def drop_duplicate_data(X: pd.DataFrame, y: pd.Series) -> tuple[pd.DataFrame, pd.Series]:
    """
    Removes duplicate rows from the feature set and synchronizes the target series.

    :param X: The input features containing potential duplicates.
    :type X: pd.DataFrame
    :param y: The target values associated with X.
    :type y: pd.Series
    :raises TypeError: If X is not a pd.DataFrame or y is not a pd.Series
    :return: A tuple of (X, y) with duplicates removed.
    :rtype: tuple[pd.DataFrame, pd.Series]
    """
    if not isinstance(X, pd.DataFrame):
        raise TypeError(f"Expected pd.DataFrame, got {type(X).__name__}")
    if not isinstance(y, pd.Series):
        raise TypeError(f"Expected pd.Series, got {type(y).__name__}")

    print("Fungsi drop_duplicate_data: parameter telah divalidasi")
    X_copy = X.copy()
    y_copy = y.copy()
    print(f"Fungsi drop_duplicate_data: shape dataset sebelum dropping duplicate adalah {X_copy.shape}")
    X_duplicate = X_copy[X_copy.duplicated()]
    print(f"Fungsi drop_duplicate_data: shape dari data yang duplicate adalah {X_duplicate.shape}")
    X_clean = (X_copy.shape[0] - X_duplicate.shape[0], X_copy.shape[1])
    print(f"Fungsi drop_duplicate_data: shape dataset setelah drop duplicate seharusnya adalah {X_clean}")
    X_copy = X_copy.drop_duplicates()
    y_copy = y_copy[X_copy.index]
    print(f"Fungsi drop_duplicate_data: shape dataset setelah dropping duplicate adalah {X_copy.shape}")
    return X_copy, y_copy

X_train, y_train = drop_duplicate_data(X_train, y_train)

Fungsi drop_duplicate_data: parameter telah divalidasi
Fungsi drop_duplicate_data: shape dataset sebelum dropping duplicate adalah (26064, 11)
Fungsi drop_duplicate_data: shape dari data yang duplicate adalah (96, 11)
Fungsi drop_duplicate_data: shape dataset setelah drop duplicate seharusnya adalah (25968, 11)
Fungsi drop_duplicate_data: shape dataset setelah dropping duplicate adalah (25968, 11)


## Handling Median Imputation

In [32]:
def median_imputation(data: pd.DataFrame, subset_data: list | dict, fit: bool) -> dict | pd.DataFrame:
    """
    Performs median calculation for fitting or fills missing values using provided medians.

    :param data: The dataset to be processed for imputation.
    :type data: pd.DataFrame
    :param subset_data: A list of columns to fit (if fit=True) or a dictionary of median values (if fit=False).
    :type subset_data: list | dict
    :param fit: Flag to determine whether to calculate new medians or apply existing ones.
    :type fit: bool
    :raises RuntimeError: If parameter types do not match the required logic for 'fit'.
    :return: A dictionary of median values if fit=True, otherwise the imputed DataFrame.
    :rtype: dict | pd.DataFrame
    """
    if not isinstance(fit, bool):
        raise RuntimeError("Fungsi median_imputation: Parameter 'fit' harus bertipe boolean.")

    if fit == True:
        if not isinstance(subset_data, list):
            raise RuntimeError("Fungsi median_imputation: Parameter subset_data harus bertipe list jika fit=True.")
    else:
        if not isinstance(subset_data, dict):
            raise RuntimeError("Fungsi median_imputation: Parameter subset_data harus bertipe dict jika fit=False.")

    print("\nFungsi median_imputation: Parameter data valid.")

    data_res = data.copy()
    subset_res = deepcopy(subset_data)

    if fit:
        imputation_data = {}
        for col in subset_res:
            median_val = data_res[col].median()
            imputation_data[col] = median_val
        print(f"Fungsi median_imputation: Hasil fitting data: {imputation_data}")
        return imputation_data
    else:
        print("Fungsi median_imputation: Jumlah missing values sebelum imputasi:")
        print(data_res.isna().sum())
        data_res.fillna(value=subset_res, inplace=True) # type: ignore
        print("\nFungsi median_imputation: Jumlah missing values setelah imputasi:")
        print(data_res.isna().sum())
        return data_res

In [33]:
subset_data = ['person_age', 'person_income', 'person_emp_length', 'loan_amnt',
               'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length']
subset_data = median_imputation(data=X_train, subset_data=subset_data, fit=True)

X_train = median_imputation(data=X_train, subset_data=subset_data, fit=False) # type: ignore
X_test = median_imputation(data=X_test, subset_data=subset_data, fit=False) # type: ignore
X_valid = median_imputation(data=X_valid, subset_data=subset_data, fit=False) # type: ignore


Fungsi median_imputation: Parameter data valid.
Fungsi median_imputation: Hasil fitting data: {'person_age': np.float64(26.0), 'person_income': np.float64(55000.0), 'person_emp_length': np.float64(4.0), 'loan_amnt': np.float64(8000.0), 'loan_int_rate': np.float64(10.99), 'loan_percent_income': np.float64(0.15), 'cb_person_cred_hist_length': np.float64(4.0)}

Fungsi median_imputation: Parameter data valid.
Fungsi median_imputation: Jumlah missing values sebelum imputasi:
person_age                       0
person_income                    0
person_home_ownership            0
person_emp_length              734
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_int_rate                 2491
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0
dtype: int64

Fungsi median_imputation: Jumlah missing values setelah imputasi:
person_age                    0
person_income              

## Encoding Categorical Data

In [34]:
def create_onehot_encoder(categories, file_path):
    """
    Creates, fits, and serializes a OneHotEncoder instance.

    :param categories: A list of categories to be used for fitting the encoder.
    :type categories: list
    :param file_path: The destination path where the fitted encoder will be serialized.
    :type file_path: str
    :raises RuntimeError: If the categories parameter is not of type list.
    :return: The fitted OneHotEncoder instance.
    :rtype: sklearn.preprocessing.OneHotEncoder
    """
    if not isinstance(categories, list):
        raise RuntimeError("Fungsi create_onehot_encoder: parameter categories harus bertipe list!")

    ohe = OneHotEncoder()
    ohe.fit(np.array(categories).reshape(-1, 1))
    utils.serialize_data(ohe, file_path) # type: ignore
    print(f"Fungsi create_onehot_encoder: kategori yang dipelajari adalah {ohe.categories_}\n")

    return ohe

def ohe_transform(dataset, subset, prefix, ohe):
    """
    Transforms a specific column in a DataFrame using a fitted OneHotEncoder.

    This function validates inputs, creates new columns with a specified prefix, 
    concatenates them to the original DataFrame, and removes the original source column.

    :param dataset: The input dataset containing the column to be transformed.
    :type dataset: pd.DataFrame
    :param subset: The name of the column within the dataset to be encoded.
    :type subset: str
    :param prefix: The prefix string to be added to the newly created one-hot columns.
    :type prefix: str
    :param ohe: A previously fitted OneHotEncoder instance.
    :type ohe: sklearn.preprocessing.OneHotEncoder
    :raises RuntimeError: If input types are incorrect or if the subset column 
        is not found in the DataFrame.
    :return: A new DataFrame with the original column replaced by one-hot encoded columns.
    :rtype: pd.DataFrame
    """
    if not isinstance(dataset, pd.DataFrame):
        raise RuntimeError("Fungsi ohe_transform: parameter dataset harus bertipe DataFrame!")
    
    if not isinstance(ohe, OneHotEncoder):
        raise RuntimeError("Fungsi ohe_transform: parameter ohe harus bertipe OneHotEncoder!")

    if not isinstance(prefix, str):
        raise RuntimeError("Fungsi ohe_transform: parameter prefix harus bertipe str!")

    if not isinstance(subset, str):
        raise RuntimeError("Fungsi ohe_transform: parameter subset harus bertipe str!")

    try:
        _ = dataset.columns.get_loc(subset)
    except:
        raise RuntimeError("Fungsi ohe_transform: parameter subset string namun data tidak ditemukan dalam daftar kolom yang terdapat pada parameter dataset.")

    print("Fungsi ohe_transform: parameter telah divalidasi.")
    dataset = dataset.copy()
    print(f"Fungsi ohe_transform: daftar nama kolom sebelum dilakukan pengkodean adalah {list(dataset.columns)}")
    col_names = [f"{prefix}_{cat}" for cat in ohe.categories_[0].tolist()] # type: ignore
    encoded = pd.DataFrame(ohe.transform(dataset[[subset]]).toarray(), # type: ignore
                           columns=col_names,
                           index=dataset.index)
    dataset = pd.concat([dataset, encoded], axis=1)
    dataset.drop(columns=[subset], inplace=True)
    print(f"Fungsi ohe_transform: daftar nama kolom setelah dilakukan pengkodean adalah {list(dataset.columns)}\n")

    return dataset

In [35]:
person_home_ownership = ['RENT', 'MORTGAGE', 'OWN', 'OTHER']
loan_intent = ['PERSONAL', 'EDUCATION', 'MEDICAL', 'VENTURE', 'HOMEIMPROVEMENT', 'DEBTCONSOLIDATION']
loan_grade = ['A', 'B', 'C', 'D', 'E', 'F', 'G']
cb_person_default_on_file = ['Y', 'N']

ohe_home_ownership = create_onehot_encoder(person_home_ownership, "models/ohe_home_ownership.pkl")
ohe_loan_intent = create_onehot_encoder(loan_intent, "models/ohe_loan_intent.pkl")
ohe_loan_grade = create_onehot_encoder(loan_grade, "models/ohe_loan_grade.pkl")
ohe_default_on_file = create_onehot_encoder(cb_person_default_on_file, "models/ohe_default_on_file.pkl")

sets = [X_train, X_test, X_valid]

for i in range(len(sets)):
    sets[i] = ohe_transform(sets[i], "person_home_ownership", "home_ownership", ohe_home_ownership)
    sets[i] = ohe_transform(sets[i], "loan_intent", "loan_intent", ohe_loan_intent)
    sets[i] = ohe_transform(sets[i], "loan_grade", "loan_grade", ohe_loan_grade)
    sets[i] = ohe_transform(sets[i], "cb_person_default_on_file", "default_onfile", ohe_default_on_file)

X_train, X_test, X_valid = sets

Data serialized on models/ohe_home_ownership.pkl.
Fungsi create_onehot_encoder: kategori yang dipelajari adalah [array(['MORTGAGE', 'OTHER', 'OWN', 'RENT'], dtype='<U8')]

Data serialized on models/ohe_loan_intent.pkl.
Fungsi create_onehot_encoder: kategori yang dipelajari adalah [array(['DEBTCONSOLIDATION', 'EDUCATION', 'HOMEIMPROVEMENT', 'MEDICAL',
       'PERSONAL', 'VENTURE'], dtype='<U17')]

Data serialized on models/ohe_loan_grade.pkl.
Fungsi create_onehot_encoder: kategori yang dipelajari adalah [array(['A', 'B', 'C', 'D', 'E', 'F', 'G'], dtype='<U1')]

Data serialized on models/ohe_default_on_file.pkl.
Fungsi create_onehot_encoder: kategori yang dipelajari adalah [array(['N', 'Y'], dtype='<U1')]

Fungsi ohe_transform: parameter telah divalidasi.
Fungsi ohe_transform: daftar nama kolom sebelum dilakukan pengkodean adalah ['person_age', 'person_income', 'person_home_ownership', 'person_emp_length', 'loan_intent', 'loan_grade', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 

/Users/bening/Code/_data_portfolios/credit-risk-classification/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but OneHotEncoder was fitted without feature names
  warnings.warn(
/Users/bening/Code/_data_portfolios/credit-risk-classification/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but OneHotEncoder was fitted without feature names
  warnings.warn(
/Users/bening/Code/_data_portfolios/credit-risk-classification/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but OneHotEncoder was fitted without feature names
  warnings.warn(
/Users/bening/Code/_data_portfolios/credit-risk-classification/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but OneHotEncoder was fitted without feature names
  warnings.warn(
/Users/bening/Code/_data_portfolios/credit-risk-classification/.

In [36]:
utils.serialize_data(X_train, "data/processed/X_train_prep.pkl") # type: ignore
utils.serialize_data(X_test, "data/processed/X_test_prep.pkl") # type: ignore
utils.serialize_data(X_valid, "data/processed/X_valid_prep.pkl") # type: ignore

Data serialized on data/processed/X_train_prep.pkl.
Data serialized on data/processed/X_test_prep.pkl.
Data serialized on data/processed/X_valid_prep.pkl.


In [37]:
utils.serialize_data(y_train, "data/processed/y_train_prep.pkl")

Data serialized on data/processed/y_train_prep.pkl.
